In [3]:
import os, json, time

In [2]:
from together import Together
import utils

key_file = 'together-lab-key.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = Together(
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [4]:
model_name = 'llama3-turbo'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'meta-llama/Llama-3.3-70B-Instruct-Turbo'

In [5]:
data_dir = '../data/final_dataset'
long_ans_files = ['certamen_translation_long.json', 
                  'junior_scholarship_translation_long.json', 
                  'prosody_caesura_scansion_english.json',
                  'prosody_caesura_scansion_latin.json',
                  'prosody_feet_questions_english.json',
                  'prosody_feet_questions_latin.json'
                  ]
long_ans_files = [os.path.join(data_dir, f) for f in long_ans_files]

file_to_data = {}
for file in long_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_translation_long.json 930
junior_scholarship_translation_long.json 350
prosody_caesura_scansion_english.json 41
prosody_caesura_scansion_latin.json 41
prosody_feet_questions_english.json 20
prosody_feet_questions_latin.json 20


In [6]:
def construct_long_ans_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    question_text += '\n' + utils.long_ans_format_instructions

    return question_text

In [8]:
prompt = construct_long_ans_user_prompt(file_to_data['certamen_translation_long.json'][0])
prompt

'Translate the motto of Alabama: Audēmus iūra nostra dēfendere.\nAt the end of your response, give your answer as:\nAnswer: answer text'

In [9]:
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.choices[0].message.content)

The motto of Alabama, "Audēmus iūra nostra dēfendere," is a Latin phrase. Let's break it down:

* "Audēmus" is the first person plural form of the verb "audēre," which means "to dare." So, "audēmus" translates to "we dare."
* "Iūra" is the nominative plural form of the noun "ius," which means "right" or "law." So, "iūra" translates to "rights" or "laws."
* "Nostra" is the possessive adjective "our" in the nominative plural form.
* "Dēfendere" is the infinitive form of the verb "dēfendō," which means "to defend."

So, when we put it all together, "Audēmus iūra nostra dēfendere" translates to "We dare defend our rights."

Answer: We dare defend our rights.


In [15]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [ ]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    for q_dict in data:
        q_id = q_dict['question_id']
        prompt = construct_long_ans_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.01)
        

        if i % 100 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)

certamen_short_answer.json
  0 / 4596
  100 / 4596
  200 / 4596
  300 / 4596
  400 / 4596
